<a href="https://colab.research.google.com/github/esemsc-db24/NewRytm/blob/main/Importing_Data_fitbit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Fitbit Data

In [1]:
! pip install fitbit
! pip install gather_keys_oauth2
! pip install fitbit requests-oauthlib CherryPy


ERROR: Could not find a version that satisfies the requirement gather_keys_oauth2 (from versions: none)
ERROR: No matching distribution found for gather_keys_oauth2


In [2]:
# mounting to my drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import fitbit
from requests_oauthlib import OAuth2Session
import webbrowser

# Replace these with your Fitbit App credentials
CLIENT_ID='23Q7KL'
CLIENT_SECRET='7f9286c713918599bb2a2e86fb73a3b5'
REDIRECT_URI = "http://127.0.0.1:8080/"

AUTH_URL = "https://www.fitbit.com/oauth2/authorize"
TOKEN_URL = "https://api.fitbit.com/oauth2/token"

# Initialize OAuth2 session
fitbit_session = OAuth2Session(CLIENT_ID, redirect_uri=REDIRECT_URI, scope=["activity", "heartrate"])
auth_url, state = fitbit_session.authorization_url(AUTH_URL)

# Open authorization URL in a browser (this might not work in Colab, so copy the URL manually)
print(f"Go to this URL to authorize Fitbit access:\n{auth_url}")
webbrowser.open(auth_url)  # You might need to copy-paste manually

# Paste the URL you get redirected to after authorization
redirect_response = input("Paste the full redirect URL here: ")

# Fetch the access token
token = fitbit_session.fetch_token(TOKEN_URL, client_secret=CLIENT_SECRET, authorization_response=redirect_response)

# Extract access & refresh tokens
ACCESS_TOKEN = token['access_token']
REFRESH_TOKEN = token['refresh_token']

print("✅ Authentication successful!")
print(f"Access Token: {ACCESS_TOKEN}")
print(f"Refresh Token: {REFRESH_TOKEN}")

# Initialize Fitbit API client
auth2_client = fitbit.Fitbit(CLIENT_ID, CLIENT_SECRET, oauth2=True,
                             access_token=ACCESS_TOKEN, refresh_token=REFRESH_TOKEN)


Go to this URL to authorize Fitbit access:
https://www.fitbit.com/oauth2/authorize?response_type=code&client_id=23Q7KL&redirect_uri=http%3A%2F%2F127.0.0.1%3A8080%2F&scope=activity+heartrate&state=2zMbI1UaG2Ub4JBZrVLV2oYo60DPWR


KeyboardInterrupt: Interrupted by user

In [4]:
import os

# Change directory to your Drive
drive_path = "/content/drive/My Drive/"  # This is your Google Drive root
os.listdir(drive_path)
import sys
sys.path.append("/content/drive/My Drive/New_rythm/python-fitbit-master")

import gather_keys_oauth2 as Oauth2
%matplotlib inline
import matplotlib.pyplot as plt
import fitbit
# gather_keys_oauth2.py file needs to be in the same directory.
# also needs to install cherrypy: https://pypi.org/project/CherryPy/
# pip install CherryPy
import pandas as pd
import datetime


# YOU NEED TO PUT IN YOUR OWN CLIENT_ID AND CLIENT_SECRET
CLIENT_ID='23Q7KL'
CLIENT_SECRET='7f9286c713918599bb2a2e86fb73a3b5'

In [ ]:
server=Oauth2.OAuth2Server(CLIENT_ID, CLIENT_SECRET)
server.browser_authorize()
ACCESS_TOKEN=str(server.fitbit.client.session.token['access_token'])
REFRESH_TOKEN=str(server.fitbit.client.session.token['refresh_token'])
auth2_client=fitbit.Fitbit(CLIENT_ID,CLIENT_SECRET,oauth2=True,access_token=ACCESS_TOKEN,refresh_token=REFRESH_TOKEN)

[02/Mar/2025:18:51:17] ENGINE Listening for SIGTERM.
INFO:cherrypy.error:[02/Mar/2025:18:51:17] ENGINE Listening for SIGTERM.
[02/Mar/2025:18:51:17] ENGINE Listening for SIGHUP.
INFO:cherrypy.error:[02/Mar/2025:18:51:17] ENGINE Listening for SIGHUP.
[02/Mar/2025:18:51:17] ENGINE Listening for SIGUSR1.
INFO:cherrypy.error:[02/Mar/2025:18:51:17] ENGINE Listening for SIGUSR1.
[02/Mar/2025:18:51:17] ENGINE Bus STARTING
INFO:cherrypy.error:[02/Mar/2025:18:51:17] ENGINE Bus STARTING
CherryPy Checker:
The Application mounted at '' has an empty config.

[02/Mar/2025:18:51:17] ENGINE Started monitor thread 'Autoreloader'.
INFO:cherrypy.error:[02/Mar/2025:18:51:17] ENGINE Started monitor thread 'Autoreloader'.
[02/Mar/2025:18:51:18] ENGINE Error in 'start' listener <bound method Server.start of <cherrypy._cpserver.Server object at 0x79ba987dfa50>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/portend.py", line 122, in free
    Checker(timeout=0.1).assert_free

In [ ]:
import datetime
oneDayData = auth2_client.intraday_time_series('activities/heart',
                                               base_date='2025-02-13',
                                               detail_level='1min',
                                               start_time="00:00",
                                               end_time="23:59")

if 'activities-heart-intraday' in oneDayData:
    df = pd.DataFrame(oneDayData['activities-heart-intraday']['dataset'])
    print("Intraday heart rate data successfully loaded!")
else:
    print("No intraday heart rate data available.")

In [ ]:
help(auth2_client.intraday_time_series)

In [ ]:
oneDayData

In [ ]:
df = pd.DataFrame(oneDayData['activities-heart-intraday']['dataset'])

In [ ]:
df

In [ ]:
# Convert 'time' to datetime (full format) but keep only the time part as a string
df["time"] = pd.to_datetime(df["time"], format="%H:%M:%S").dt.strftime("%H:%M:%S")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df["time"], df["value"], linestyle='-', label="Heart Rate")

# Formatting
plt.xlabel("Time")
plt.ylabel("Heart Rate (BPM)")
plt.title("Heart Rate Over Time")
plt.xticks(df["time"][::50], rotation=45) # Rotate x-axis for better readability
plt.legend()
plt.grid(True)

# Show plot
plt.show()
